# {{dataset_id}} in your browser

This notebook runs Python in this browser tab with [Pyodide](https://pyodide.org/), and reads NEMAR's Zarr copy of [{{dataset_id}}]({{dataset_page_base}}/dataset/{{dataset_id}}).
That copy is lossy and may be downsampled, so download the dataset for anything that needs the original files.

**Before you start**

- The setup cell below runs by itself when the notebook opens. Wait for its "Ready" line.
- Then run the cells in order: Shift+Enter runs one cell, and Run > Run All Cells runs them all.
- Need another package? Add a cell with `%pip install pandas` and run it. The setup cell already installs SciPy; pandas, scikit-learn and others install in seconds, and other pure-Python packages, such as seaborn, come from the Python Package Index (PyPI).
- Your edits are saved in this browser only. Use File > Download to keep a copy.

In [ ]:
# Setup: runs by itself when the notebook opens.
%pip install eegprep-lean scipy
%matplotlib inline
import eegprep_lean

print("Ready: eegprep-lean", eegprep_lean.__version__)

In [ ]:
index = await eegprep_lean.read_index(
    "{{dataset_id}}", index_url="{{zarr_base}}/{{dataset_id}}/zarr/index.json"
)
print(index.store_count, "recordings with a Zarr copy")
for store in index.stores[:10]:
    print(store.path)

In [ ]:
store = index.stores[0]
group = store.group()
seconds = 2
window = await eegprep_lean.read_window(
    index,
    store,
    group=group,
    start_sample=0,
    n_samples=int(seconds * group.rate),
    channels=list(range(min(4, group.n_channels))),
)
print(window.data.shape, window.unit, window.rate, (window.labels or ())[:8])
eegprep_lean.plot_window(window).figure

## The power spectrum

The first two minutes of every channel, or the whole recording if it is shorter: enough for a stable spectrum, without downloading the rest.
Welch's method, with SciPy: the average of Hann-tapered periodograms over 2-second segments that overlap by half. Each thin line is a channel, and the black line is their median.
Electrooculography (EOG), electrocardiography (ECG) and electromyography (EMG) channels are left out by name when the group also holds other channels.
To look at another recording, change `store` in the cell above and run both cells again.

More examples, among them an event-related potential (ERP) image of a dataset's conditions worked through on ERP CORE (nm000132), are in NEMAR's documentation: [docs.nemar.org/platform/zarr/examples](https://docs.nemar.org/platform/zarr/examples/).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import periodogram

seconds = min(120, group.duration_s)
first_minutes = await eegprep_lean.read_window(
    index, store, group=group, start_sample=0, n_samples=int(seconds * group.rate)
)
if first_minutes.data.shape[1] < 2:
    raise ValueError(
        f"{store.path} has no samples to take a spectrum of; choose another recording."
    )
names = [str(name) for name in (first_minutes.labels or range(first_minutes.data.shape[0]))]
others = ("EOG", "ECG", "EKG", "EMG")
keep = [i for i, name in enumerate(names) if not any(tag in name.upper() for tag in others)]
keep = keep or list(range(len(names)))
rate = first_minutes.rate
# Welch's method one segment at a time: scipy.signal.welch takes a strided view of
# every segment at once, which is too large for this browser's 32-bit Python on a
# long read of many channels, although the result is the same.
segment = min(int(2 * rate), first_minutes.data.shape[1])
starts = range(0, first_minutes.data.shape[1] - segment + 1, max(1, segment // 2))
data = first_minutes.data[keep]
power = np.mean(
    [periodogram(data[:, s : s + segment], fs=rate, window="hann")[1] for s in starts], axis=0
)
freqs = np.fft.rfftfreq(segment, 1 / rate)
# 80 Hz covers the rhythms of brain recordings; any other modality shows its whole band.
top = min(80.0, rate / 2) if group.modality.upper() in ("EEG", "MEG", "IEEG") else rate / 2
band = (freqs >= 0.5) & (freqs <= top)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs[band], power[:, band].T, lw=0.5, alpha=0.4)
ax.semilogy(freqs[band], np.median(power[:, band], axis=0), color="k", lw=1.5, label="median")
ax.set(
    xlabel="Frequency (Hz)",
    ylabel=f"Power ({first_minutes.unit}\u00b2/Hz)" if first_minutes.unit else "Power per Hz",
    title=f"{store.path}\n{len(keep)} channels, first {seconds:.0f} s",
)
ax.legend()
plt.show()
left_out = [names[i] for i in range(len(names)) if i not in keep]
print(len(keep), "of", len(names), "channels; left out:", left_out)